# Your first agent

Assistant version 1. Read [the module guide](README.md) for objectives, diagrams, costs and exercises.

Run all cells in order. These cells replay synthetic fixtures and block external traffic. They do not measure live model quality. Each example's code is editable below; its original command remains in the docstring. Use the CLI without `--mock` only after reviewing provider setup and cost.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_hello_agent

Source: [01_hello_agent.py](examples/01_hello_agent.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Agent defaults inspected safely; configured Agent, system_prompt and synchronous invocation.

Run: uv run python modules/01_first_agent/examples/01_hello_agent.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent
from strands.models.bedrock import DEFAULT_BEDROCK_MODEL_ID

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli


def main() -> None:
    _, cfg = lesson_settings()
    # Inspect the SDK constant instead of accidentally invoking its paid default.
    console.print("SDK default provider: Bedrock; model constant:", DEFAULT_BEDROCK_MODEL_ID)
    console.print("Our configured model:", cfg.model_id or "scripted")
    # KEY IDEA: explicit selection keeps a future SDK default from changing our bill.
    agent = Agent(
        model=get_model(cfg, responses=["Welcome to Nimbus Travel."]),
        system_prompt="You are Nimbus's concise travel assistant.",
        callback_handler=None,
        retry_strategy=None,
    )
    complete(agent("Welcome a traveler.", limits={"turns": 2}), cfg)


run_lesson(main, "modules/01_first_agent/examples/01_hello_agent.py")

## 02_result_and_messages

Source: [02_result_and_messages.py](examples/02_result_and_messages.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""AgentResult message, stop_reason, metrics, and agent.messages.

Run: uv run python modules/01_first_agent/examples/02_result_and_messages.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli


def main() -> None:
    _, cfg = lesson_settings()
    agent = Agent(
        model=get_model(cfg, responses=["Tell me your destination."]),
        callback_handler=None,
        retry_strategy=None,
    )
    result = agent("Help me plan a Nimbus trip.", limits={"turns": 2})
    # KEY IDEA: an AgentResult is more than the printable answer.
    console.print("stop_reason:", result.stop_reason)
    console.print("answer blocks:", result.message["content"])
    console.print("history roles:", [m["role"] for m in agent.messages])
    assert agent.messages[-1]["role"] == "assistant"
    complete(result, cfg)


run_lesson(main, "modules/01_first_agent/examples/02_result_and_messages.py")

## 03_callbacks

Source: [03_callbacks.py](examples/03_callbacks.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Default printing, null_callback_handler, and custom callback handler.

Run: uv run python modules/01_first_agent/examples/03_callbacks.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent
from strands.handlers.callback_handler import null_callback_handler

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli


def main() -> None:
    _, cfg = lesson_settings()
    collected = []

    def capture(**event) -> None:
        if "data" in event:
            collected.append(event["data"])

    # KEY IDEA: callbacks observe output; they do not replace the model or tool loop.
    for label, callback in [
        ("default", None),
        ("quiet", null_callback_handler),
        ("custom", capture),
    ]:
        kwargs = {} if label == "default" else {"callback_handler": callback}
        agent = Agent(
            model=get_model(cfg, responses=["Nimbus can help."]), retry_strategy=None, **kwargs
        )
        agent("Say Nimbus can help.", limits={"turns": 2})
        console.print("Callback mode:", label)
    console.print("Captured text:", "".join(collected))
    assert collected
    complete()


run_lesson(main, "modules/01_first_agent/examples/03_callbacks.py")

## 04_async_call

Source: [04_async_call.py](examples/04_async_call.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Async/await primer and invoke_async.

Run: uv run python modules/01_first_agent/examples/04_async_call.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

import asyncio

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import run_cli


async def answer(cfg):
    agent = Agent(
        model=get_model(cfg, responses=["Nimbus async reply."]),
        callback_handler=None,
        retry_strategy=None,
    )
    # KEY IDEA: await lets other coroutines run while the model is waiting.
    return await agent.invoke_async("Give a short Nimbus welcome.", limits={"turns": 2})


def main() -> None:
    _, cfg = lesson_settings()
    complete(asyncio.run(answer(cfg)), cfg)


run_lesson(main, "modules/01_first_agent/examples/04_async_call.py")

## 05_follow_the_loop

Source: [05_follow_the_loop.py](examples/05_follow_the_loop.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Agent/event loop; model decision, tool execution, continuation and stop.

Run: uv run python modules/01_first_agent/examples/05_follow_the_loop.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent, tool

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.testing import tool_message


@tool
def refund_window() -> str:
    """Read the short synthetic cancellation condition."""
    console.print("2. Tool executed")
    return "24 hours after booking AND departure at least 7 days after creation."


def main() -> None:
    _, cfg = lesson_settings()
    model = get_model(
        cfg,
        responses=[
            tool_message("refund_window", {}),
            "Both the 24-hour and 7-day conditions apply.",
        ],
    )
    agent = Agent(model=model, tools=[refund_window], callback_handler=None, retry_strategy=None)
    console.print("1. Model receives request and may ask for a tool")
    # KEY IDEA: Strands feeds the toolResult back into another model turn.
    result = agent("Read and explain the refund window.", limits={"turns": 3})
    console.print(
        "3. Tool result in history:",
        any("toolResult" in block for msg in agent.messages for block in msg["content"]),
    )
    complete(result, cfg)


run_lesson(main, "modules/01_first_agent/examples/05_follow_the_loop.py")

## faq_responder

Source: [faq_responder.py](use_case/faq_responder.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Answer Nimbus FAQs from one policy and abstain when evidence is missing.

Run: uv run python modules/01_first_agent/use_case/faq_responder.py --mock
Expected: a Nimbus result followed by Lesson complete.
Cost: mock/local $0 hosted inference; hosted mode uses model tokens.
AWS resources created: none. Concepts: compose this module's taught APIs.
"""

from common.assistant import faq_agent
from common.lessons import complete, lesson_settings
from common.pretty import console, run_cli


def main() -> None:
    _, cfg = lesson_settings()
    replies = [
        "A full refund needs cancellation within 24 hours and departure at least 7 days "
        "after creation (POL-REF-001).",
        "I do not know; the supplied policy does not cover visas.",
    ]
    agent = faq_agent(cfg, replies)
    # KEY IDEA: supplying a policy constrains the task, but grounding must still be evaluated.
    for question in ["When can I receive a full refund?", "Which visa do I need?"]:
        console.print("Traveler:", question)
        complete(agent(question, limits={"turns": 2}), cfg)


run_lesson(main, "modules/01_first_agent/use_case/faq_responder.py")

## Practice next

Complete the three exercises in the [module guide](README.md#exercises), then answer the [quiz](quiz.md). Run the same acceptance tests against your edited starters.
